# 1. Data Preparation

This notebook constitutes the **Data Preparation** phase of the CRISP-DM methodology.

The purpose of this stage is to transform the raw OSMI Mental Health in Technology-related Jobs survey data into analytically appropriate representations for the subsequent unsupervised modelling stages.

The preparation process follows five principal activities:

1. **Data Cleaning** — identify and correct invalid, inconsistent, implausible, duplicated, or structurally inappropriate observations.
2. **Preprocessing** — convert variables into consistent numerical and categorical representations suitable for machine-learning algorithms.
3. **Feature Engineering** — derive analytically meaningful variables from the survey structure, including multi-response and conditional variables.
4. **Feature Selection** — reduce redundant or uninformative variables while preserving information relevant to the unsupervised analysis.
5. **Dimensionality Reduction** — investigate lower-dimensional representations, principally through Principal Component Analysis (PCA), while retaining complementary methods where justified.

The preparation decisions are evidence-driven and are based on the data-quality findings established during Business and Data Understanding.

No clustering algorithm is trained in this notebook. The objective is to construct and validate the analytical representations that will subsequently be supplied to the clustering experiments.

## 1.1 Preparation Objectives

The data-preparation stage has the following objectives:

- preserve the maximum amount of valid respondent information;
- distinguish genuine missingness from survey-branching missingness;
- remove demonstrably invalid or analytically inappropriate values without unnecessarily deleting respondents;
- preserve the semantic meaning of responses such as `Maybe`, `Don't know`, and `Not applicable`;
- transform multi-response survey fields into machine-readable representations;
- distinguish ordinal variables from nominal variables;
- prevent high-cardinality free text from entering the numerical clustering matrix;
- control excessive dimensionality and redundant encoded variables;
- standardize numerical representations before distance-based analysis;
- investigate PCA as the principal dimensionality-reduction representation;
- retain suitable complementary representations for later methodological comparison;
- produce clearly defined datasets for clustering and post-clustering profiling.

The final analytical matrix must therefore be **numerically valid, reproducible, documented, and free from avoidable leakage of survey structure into the clustering representation**.

## 1.2 Imports and Reproducibility

In [3]:
from __future__ import annotations

import re
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from scipy.stats import chi2_contingency
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.manifold import LocallyLinearEmbedding, MDS
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

sns.set_theme(style="whitegrid")

print(f"Random state: {RANDOM_STATE}")
print(f"pandas version: {pd.__version__}")

Random state: 42
pandas version: 3.0.6


The notebook establishes a fixed random state so that stochastic dimensionality-reduction procedures and any later stochastic modelling remain reproducible.

The imports are deliberately limited to tools required for the documented preparation workflow. The preparation stage uses statistical testing where it provides evidence about data structure, while machine-learning transformations are implemented using established scikit-learn preprocessing components.

## 1.3 Project Paths and Data Loading

In [4]:
def find_project_root(start: Path | None = None) -> Path:
    """
    Locate the project root by searching parent directories for pyproject.toml.

    Parameters
    ----------
    start:
        Directory from which the search begins.

    Returns
    -------
    Path
        Project root directory.

    Raises
    ------
    FileNotFoundError
        If no project root can be identified.
    """
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing pyproject.toml."
    )


PROJECT_ROOT = find_project_root()

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "mental-heath-in-tech-2016_20161114.csv.zip"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw dataset: {RAW_DATA_PATH}")

Project root: C:\Users\User\Documents\Projects\mental-health-in-tech-ml
Raw dataset: C:\Users\User\Documents\Projects\mental-health-in-tech-ml\data\raw\mental-heath-in-tech-2016_20161114.csv.zip


In [5]:
with zipfile.ZipFile(RAW_DATA_PATH) as archive:
    csv_members = [
        member
        for member in archive.namelist()
        if member.lower().endswith(".csv")
    ]

    if len(csv_members) != 1:
        raise ValueError(
            f"Expected exactly one CSV file in the archive, found {len(csv_members)}."
        )

    with archive.open(csv_members[0]) as file:
        df_raw = pd.read_csv(file)

print(f"Dataset shape: {df_raw.shape}")
print(f"Number of observations: {df_raw.shape[0]:,}")
print(f"Number of variables: {df_raw.shape[1]:,}")

Dataset shape: (1433, 63)
Number of observations: 1,433
Number of variables: 63


The raw survey contains **1,433 observations and 63 variables**.

The dataset is retained in its original form as `df_raw`. This separation is important because all subsequent transformations should be traceable back to the original survey responses. The raw dataframe is therefore treated as an immutable reference rather than being overwritten during preparation.

## 1.4 Raw Data Integrity Check

In [8]:
integrity_summary = pd.DataFrame(
    {
        "Metric": [
            "Rows",
            "Columns",
            "Exact duplicate rows",
            "Total missing cells",
            "Missing-cell percentage",
        ],
        "Value": [
            len(df_raw),
            df_raw.shape[1],
            df_raw.duplicated().sum(),
            int(df_raw.isna().sum().sum()),
            df_raw.isna().mean().mean() * 100,
        ],
    }
)

integrity_summary

,Metric,Value
0,Rows,1433.000000
1,Columns,63.000000
2,Exact duplicate rows,0.000000
3,Total missing cells,21960.000000
4,Missing-cell percentage,24.324594


There are **no exact duplicate rows** in the raw dataset. This indicates that the dataset does not contain duplicate records that can be identified through complete-row equality.

The presence of missing values is expected from the survey structure and must not be interpreted as evidence of defective records by itself. In particular, several survey questions were conditional on previous responses, meaning that some missing values represent **structural non-applicability rather than missing information**.

Consequently, missingness will be analysed by variable and by survey branch rather than treated uniformly.

## 1.5 Age Quality and Analytical Population

Age is a numerical variable that directly affects distance-based algorithms and dimensionality reduction.

The raw data contain implausible or highly unusual self-reported ages. Because clustering relies on numerical geometry, extreme erroneous values can have disproportionate influence after scaling and can distort principal components.

The analytical population is therefore restricted to respondents aged **18–65 inclusive**.

This boundary is not intended to claim that technology workers above 65 do not exist. It defines the study population for this survey analysis while removing only eight observations from 1,433 records.

In [9]:
age_col = "What is your age?"

age_summary = df_raw[age_col].describe().to_frame("Age")
age_summary

,Age
count,1433.000000
mean,34.286113
std,11.290931
min,3.000000
25%,28.000000
50%,33.000000
75%,39.000000
max,323.000000


The raw age variable ranges from **3 to 323 years**, while the median is **33 years**.

The values at the extreme ends of the distribution cannot reasonably be treated as ordinary observations for an analysis of technology-sector working-age respondents. The distribution therefore requires explicit quality control before age is scaled or incorporated into a multivariate representation.

In [10]:
age_mask = df_raw[age_col].between(18, 65, inclusive="both")

print(f"Observations within 18–65: {age_mask.sum():,}")
print(f"Observations outside 18–65: {(~age_mask).sum():,}")
print(f"Retention rate: {age_mask.mean():.2%}")

display(
    df_raw.loc[~age_mask, [age_col]]
    .sort_values(age_col)
    .reset_index(drop=True)
)

Observations within 18–65: 1,425
Observations outside 18–65: 8
Retention rate: 99.44%


,What is your age?
0,3
1,15
2,17
3,66
4,70
5,74
6,99
7,323


The 18–65 rule retains **1,425 of 1,433 observations (99.44%)** and excludes only eight records.

The excluded ages are:

- 3
- 15
- 17
- 66
- 70
- 74
- 99
- 323

The restriction therefore has a very small effect on sample size while removing the extreme and sparsely represented tail of the self-reported age distribution.

This is preferable to replacing implausible ages with the median because the values are not missing observations; they are observations whose reported values fall outside the defined analytical population.

In [11]:
df = df_raw.loc[age_mask].copy()

print(f"Prepared working dataset shape after age restriction: {df.shape}")

Prepared working dataset shape after age restriction: (1425, 63)


The working dataset now contains **1,425 respondents and 63 variables**.

No other observations have been removed at this point. This is intentional: subsequent cleaning decisions will be made variable-by-variable so that valid information is not lost unnecessarily.

## 1.6 Row-Level Missingness

In [12]:
row_missing_rate = df.isna().mean(axis=1)

row_missing_summary = pd.Series(
    {
        "Minimum missingness": row_missing_rate.min(),
        "Median missingness": row_missing_rate.median(),
        "Mean missingness": row_missing_rate.mean(),
        "Maximum missingness": row_missing_rate.max(),
        "Rows with >50% missing": (row_missing_rate > 0.50).sum(),
    }
)

display(row_missing_summary.to_frame("Value"))

,Value
Minimum missingness,0.142857
Median missingness,0.222222
Mean missingness,0.243297
Maximum missingness,0.571429
Rows with >50% missing,19.000000


Only a small number of respondents exceed the 50% row-level missingness threshold.

The dataset therefore does not support a blanket rule that respondents with substantial missingness should be deleted. More importantly, a large proportion of missing values arise from conditional survey questions.

The preparation strategy is consequently **variable- and structure-aware rather than row-deletion-based**. Respondents are retained unless there is a specific evidence-based reason that their observations cannot contribute to the analytical representation.

## 1.7 Structural Missingness

Several employer-related variables are not applicable to respondents who identified themselves as self-employed.

This represents **structural missingness**:

$$
M_j =
\begin{cases}
1, & \text{if the variable is structurally not applicable} \\
0, & \text{otherwise}
\end{cases}
$$

Such values must not be replaced with the mode of the observed population because doing so would falsely assign employer characteristics to self-employed respondents.

The first task is therefore to verify the relationship statistically.

In [13]:
self_employed_col = "Are you self-employed?"

employer_columns = [
    "How many employees does your company or organization have?",
    "Is your employer primarily a tech company/organization?",
    "Does your employer provide mental health benefits as part of healthcare coverage?",
    "Do you know the options for mental health care available under your employer-provided coverage?",
    "Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?",
    "Does your employer offer resources to learn more about mental health concerns and options for seeking help?",
    "Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?",
    "If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:",
    "Do you think that discussing a mental health disorder with your employer would have negative consequences?",
]

for column in employer_columns:
    missing_table = pd.crosstab(
        df[self_employed_col],
        df[column].isna(),
        margins=False,
    )

    chi2, p_value, _, _ = chi2_contingency(missing_table)

    print(f"\n{column}")
    print(f"Chi-square statistic: {chi2:.3f}")
    print(f"p-value: {p_value:.6g}")
    display(missing_table)


How many employees does your company or organization have?
Chi-square statistic: 1418.773
p-value: 1.74981e-310


How many employees does your company or organization have?,False,True
Are you self-employed?,,
0,1139,0
1,0,286



Is your employer primarily a tech company/organization?
Chi-square statistic: 1418.773
p-value: 1.74981e-310


Is your employer primarily a tech company/organization?,False,True
Are you self-employed?,,
0,1139,0
1,0,286



Does your employer provide mental health benefits as part of healthcare coverage?
Chi-square statistic: 1418.773
p-value: 1.74981e-310


...

Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?,False,True
Are you self-employed?,,
0,1139,0
1,0,286



If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:
Chi-square statistic: 1418.773
p-value: 1.74981e-310


"If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:",False,True
Are you self-employed?,,
0,1139,0
1,0,286



Do you think that discussing a mental health disorder with your employer would have negative consequences?
Chi-square statistic: 1418.773
p-value: 1.74981e-310


Do you think that discussing a mental health disorder with your employer would have negative consequences?,False,True
Are you self-employed?,,
0,1139,0
1,0,286


The missingness pattern of the employer-specific variables is strongly aligned with self-employment status.

The chi-square test evaluates:

$$
H_0:\text{missingness is independent of self-employment status}
$$

against:

$$
H_1:\text{missingness is associated with self-employment status}
$$

A statistically significant result provides evidence that these missing values are not random omissions. They correspond to the survey's respondent structure.

Accordingly, these fields require explicit treatment as **Not Applicable (Self-Employed)** rather than ordinary missing-value imputation.

## 1.8 Semantic Response States

The survey contains responses such as:

- `Don't know`
- `Maybe`
- `Not sure`
- `Not applicable`
- `Not applicable to me`

These responses are not equivalent to missing values.

For example, `Maybe` represents uncertainty, whereas `No` represents a negative response. Similarly, `Don't know` indicates lack of knowledge rather than absence of the underlying condition.

These states will therefore remain explicit categorical states unless a variable-specific encoding requires them to be represented separately from an ordinal scale.

In [14]:
semantic_states = {
    "Don't know": 0,
    "Maybe": 0,
    "Not sure": 0,
    "Not applicable": 0,
    "Not applicable to me": 0,
}

semantic_summary = []

for state in semantic_states:
    matches = 0
    for column in df.select_dtypes(include=["object"]).columns:
        matches += df[column].astype("string").eq(state).sum()

    semantic_summary.append(
        {
            "Response state": state,
            "Occurrences": int(matches),
        }
    )

display(pd.DataFrame(semantic_summary))

,Response state,Occurrences
0,Don't know,0
1,Maybe,4591
2,Not sure,0
3,Not applicable,0
4,Not applicable to me,1392
